# Phase 5.6 — End-to-End Validation on Colab T4

**AI Public Speaking Assistant — Full Pipeline Test**

This notebook runs the complete pipeline (3 modalities → fusion → scoring → report) on 3 test videos.

| Test | Video | Expected |
|------|-------|----------|
| A | Good speaker (TED talk) | Most scores 70-90 |
| B | Nervous speaker (student) | Some scores 40-60 |
| C | Monotone speaker (lecture) | Expressiveness low |

**Runtime:** T4 GPU required. Estimated ~5-10 min per video.

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Change runtime to T4 GPU.")

In [ ]:
# Cell 2: Install all dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets librosa praat-parselmouth speechbrain noisereduce soundfile
!pip install -q opencv-python mediapipe ultralytics
!pip install -q scikit-learn scipy pyyaml tqdm
!pip install -q sentence-transformers spacy
!pip install -q language-tool-python nltk textstat
!pip install -q matplotlib weasyprint
!python -m spacy download en_core_web_sm -q

# NLTK data
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('cmudict', quiet=True)

print("\n✓ All dependencies installed")

In [ ]:
# Cell 3: Mount Drive + copy models/videos locally (source code stays on Drive)
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DRIVE_PROJECT = '/content/drive/MyDrive/Claude-assistant'
LOCAL_PROJECT = '/content/Claude-assistant'

if not os.path.exists(os.path.join(DRIVE_PROJECT, 'src', 'master_pipeline.py')):
    raise FileNotFoundError("Project not found on Drive at " + DRIVE_PROJECT)

# Create local project structure
os.makedirs(LOCAL_PROJECT, exist_ok=True)

# Symlink source code + configs (fast, read-only)
for folder in ['src', 'configs', 'tests', 'prompts']:
    link = os.path.join(LOCAL_PROJECT, folder)
    target = os.path.join(DRIVE_PROJECT, folder)
    if os.path.exists(target) and not os.path.exists(link):
        os.symlink(target, link)
        print(f"  ✓ Linked {folder}/")

# Copy models locally (MediaPipe needs local filesystem)
local_models = os.path.join(LOCAL_PROJECT, 'models')
if not os.path.exists(local_models):
    print("  Copying models/ to local disk (one-time, ~400MB)...")
    shutil.copytree(os.path.join(DRIVE_PROJECT, 'models'), local_models)
    print(f"  ✓ Models copied")
else:
    print(f"  ✓ Models already local")

# Copy test videos locally (faster I/O for video processing)
local_data = os.path.join(LOCAL_PROJECT, 'data', 'raw')
os.makedirs(local_data, exist_ok=True)
videos = {
    "good": "test_good_speaker_5min.mp4",
    "nervous": "test_nervous_speaker_5min.mp4",
    "monotone": "test_monotone_speaker_5min.mp4",
}
for name, fname in videos.items():
    src = os.path.join(DRIVE_PROJECT, 'data', 'raw', fname)
    dst = os.path.join(local_data, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        size = os.path.getsize(dst) / 1e6
        print(f"  ✓ {name}: {size:.1f} MB copied")
    elif os.path.exists(dst):
        size = os.path.getsize(dst) / 1e6
        print(f"  ✓ {name}: {size:.1f} MB (already local)")
    else:
        print(f"  ✗ {name}: NOT FOUND on Drive")

# Create local dirs for outputs
os.makedirs(os.path.join(LOCAL_PROJECT, 'data', 'frames'), exist_ok=True)
os.makedirs(os.path.join(LOCAL_PROJECT, 'data', 'outputs'), exist_ok=True)
os.makedirs(os.path.join(LOCAL_PROJECT, 'reports'), exist_ok=True)

os.chdir(LOCAL_PROJECT)
print(f"\n✓ Working directory: {os.getcwd()}")

In [ ]:
# Cell 3b: Re-encode test videos to H.264 for OpenCV compatibility
# Colab's OpenCV can't decode VP9/H.265 frames — ffmpeg re-encodes to H.264
import subprocess, os

video_dir = 'data/raw'
for fname in sorted(os.listdir(video_dir)):
    if not fname.endswith('.mp4'):
        continue
    src = os.path.join(video_dir, fname)
    tmp = src.replace('.mp4', '_h264.mp4')

    # Check if already H.264 by trying to read a frame
    import cv2
    cap = cv2.VideoCapture(src)
    ret, _ = cap.read()
    cap.release()
    if ret:
        print(f'  ✓ {fname} — already readable, skipping')
        continue

    print(f'  Re-encoding {fname} to H.264...')
    result = subprocess.run(
        ['ffmpeg', '-i', src, '-c:v', 'libx264', '-preset', 'fast',
         '-c:a', 'aac', '-y', tmp],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        os.replace(tmp, src)
        print(f'  ✓ {fname} re-encoded')
    else:
        print(f'  ✗ {fname} FAILED: {result.stderr[-200:]}')

# Verify
print()
for fname in sorted(os.listdir(video_dir)):
    if not fname.endswith('.mp4'):
        continue
    cap = cv2.VideoCapture(os.path.join(video_dir, fname))
    ret, frame = cap.read()
    cap.release()
    status = f'read=True, shape={frame.shape}' if ret else 'read=False'
    print(f'  {fname}: {status}')


In [ ]:
# Cell 4: Set environment + Java for LanguageTool
import os, sys
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

# Install Java if not present (needed for grammar checking)
!apt-get install -qq -y default-jdk > /dev/null 2>&1
!java -version 2>&1 | head -1

# Add project to path
sys.path.insert(0, '/content/Claude-assistant')

# Enable logging so we can see module loading
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

from src.fusion.fusion_engine import FusionEngine
from src.scoring.dimension_scorer import DimensionScorer
from src.report.coaching_writer import CoachingWriter
from src.report.chart_generator import ChartGenerator
from src.report.report_builder import ReportBuilder
from src.master_pipeline import SpeechCoachPipeline

print("✓ All pipeline imports successful")

In [ ]:
# Cell 5: Run unit tests first (quick sanity check)
!cd /content/Claude-assistant && python -m pytest tests/fusion/ tests/scoring/ tests/report/test_coaching_writer.py -v --tb=short 2>&1 | tail -20

## Step-by-Step Pipeline Execution

We run each modality separately to capture intermediate outputs, then fuse + score + report.
This gives better visibility than running the master pipeline as a black box.

In [ ]:
# Cell 5b: Download fresh MediaPipe models + diagnose body pipeline
import os, urllib.request

# ── Download MediaPipe task files matching THIS Colab's version ──
print("Downloading fresh MediaPipe models for Colab compatibility...")
MODELS = {
    "pose_landmarker_lite.task": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task",
    "face_landmarker.task": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task",
    "hand_landmarker.task": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
}

model_dir = os.path.join(os.getcwd(), "models")
for filename, url in MODELS.items():
    path = os.path.join(model_dir, filename)
    print(f"  {filename}...", end=" ")
    urllib.request.urlretrieve(url, path)
    print(f"OK ({os.path.getsize(path):,} bytes)")

# ── Verify models load ──
print("\nVerifying MediaPipe model loading...")
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

for name, filename in [("Pose", "pose_landmarker_lite.task"),
                        ("Face", "face_landmarker.task"),
                        ("Hand", "hand_landmarker.task")]:
    path = os.path.join(model_dir, filename)
    try:
        opts_cls = getattr(vision, f"{name}LandmarkerOptions")
        det_cls = getattr(vision, f"{name}Landmarker")
        opts = opts_cls(
            base_options=mp_python.BaseOptions(model_asset_path=path),
            running_mode=vision.RunningMode.IMAGE,
        )
        det = det_cls.create_from_options(opts)
        det.close()
        print(f"  {name}Landmarker: ✓ LOADS OK")
    except Exception as e:
        print(f"  {name}Landmarker: ✗ FAILED — {e}")

# ── Check body pipeline module loading ──
print("\nBody pipeline module check...")
from src.body.pipeline import BodyAnalysisPipeline
body_test = BodyAnalysisPipeline()
expected = ["frame_extractor", "person_detector", "pose_estimator",
            "posture_scorer", "gesture_classifier", "hand_tracker",
            "gaze_estimator", "facial_emotion", "stage_movement", "output_assembler"]
loaded = 0
for mod in expected:
    status = "✓" if mod in body_test._modules else "✗"
    if mod in body_test._modules:
        loaded += 1
    print(f"  {status} {mod}")
print(f"\n{loaded}/{len(expected)} modules loaded")

# ── Quick video read test ──
import cv2
test_vid = "data/raw/test_good_speaker_5min.mp4"
if os.path.exists(test_vid):
    cap = cv2.VideoCapture(test_vid)
    print(f"\nVideo test: opened={cap.isOpened()}, fps={cap.get(cv2.CAP_PROP_FPS)}, frames={int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}")
    ret, frame = cap.read()
    print(f"  First frame: read={ret}, shape={frame.shape if ret else 'N/A'}")
    cap.release()

In [ ]:
# Cell 5c: Quick body pipeline debug — isolate where it fails on Colab
import os, cv2, traceback

video = "data/raw/test_good_speaker_5min.mp4"
print(f"CWD: {os.getcwd()}")
print(f"Video exists: {os.path.exists(video)} ({os.path.getsize(video)/1e6:.1f} MB)" if os.path.exists(video) else f"Video MISSING: {video}")

# Test OpenCV can read the video
cap = cv2.VideoCapture(video)
print(f"VideoCapture opened: {cap.isOpened()}")
if cap.isOpened():
    ret, frame = cap.read()
    print(f"First frame: ret={ret}, shape={frame.shape if ret else 'N/A'}")
    cap.release()

# Test frame extraction
print("\nTesting frame extraction...")
try:
    from src.body.frame_extractor import extract_frames
    result = extract_frames(video, output_dir="data/frames/test_debug", target_fps=5)
    print(f"Extracted {result['total_frames']} frames to {result['frame_dir']}")
    print(f"First frame exists: {os.path.exists(result['frame_paths'][0])}")
except Exception as e:
    print(f"Frame extraction FAILED: {e}")
    traceback.print_exc()

# Test body pipeline process() with verbose error catching
print("\nTesting body pipeline process()...")
try:
    import logging
    logging.basicConfig(level=logging.INFO, format='%(name)s: %(message)s', force=True)
    from src.body.pipeline import BodyAnalysisPipeline
    pipe = BodyAnalysisPipeline()
    print(f"Modules: {len(pipe._modules)}/10")
    result = pipe.process(video)
    print(f"Segments: {len(result.get('segments', []))}")
    print(f"Result keys: {list(result.keys())}")
except Exception as e:
    print(f"Body pipeline FAILED: {e}")
    traceback.print_exc()

In [ ]:
# Cell 6: Helper — run full pipeline on one video with validation
import time, json, os, traceback
import torch

def run_full_pipeline(video_path, label, output_base="reports"):
    """Run all 3 modalities + fusion + scoring + report on a single video."""
    output_dir = os.path.join(output_base, label)
    os.makedirs(output_dir, exist_ok=True)
    results = {"label": label, "video": video_path, "errors": []}
    t0 = time.time()

    # ── Modality 1: Voice ──────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"[{label.upper()}] Processing: {os.path.basename(video_path)}")
    print(f"{'='*60}")

    try:
        print("\n[1/8] Voice analysis...")
        t1 = time.time()
        from src.pipeline import VoiceAnalysisPipeline
        voice_pipe = VoiceAnalysisPipeline()
        voice_output = voice_pipe.process(video_path)
        voice_time = time.time() - t1
        print(f"  ✓ Voice done in {voice_time:.0f}s")
        print(f"  Windows: {len(voice_output.get('windows', []))}")
        print(f"  Duration: {voice_output.get('summary', {}).get('duration_sec', 0):.1f}s")
        # Check transcript
        transcript = voice_output.get("transcript", {})
        word_count = len(transcript.get("words", [])) if isinstance(transcript, dict) else 0
        print(f"  Transcript words: {word_count}")
        results["voice"] = {
            "status": "OK",
            "time_sec": voice_time,
            "window_count": len(voice_output.get("windows", [])),
            "duration_sec": voice_output.get("summary", {}).get("duration_sec", 0),
            "wpm": voice_output.get("summary", {}).get("words_per_minute", 0),
            "filler_count": voice_output.get("summary", {}).get("filler_count", 0),
            "transcript_words": word_count,
        }
    except Exception as e:
        print(f"  ✗ Voice FAILED: {e}")
        traceback.print_exc()
        voice_output = {"windows": [], "summary": {"duration_sec": 300}, "transcript": {}}
        results["voice"] = {"status": "FAILED", "error": str(e)}
        results["errors"].append(f"Voice: {e}")

    torch.cuda.empty_cache()

    # ── Modality 2: Body ───────────────────────────────────────
    try:
        print("\n[2/8] Body language analysis...")
        t2 = time.time()
        from src.body.pipeline import BodyAnalysisPipeline
        body_pipe = BodyAnalysisPipeline()
        # Print loaded modules for debugging
        loaded_mods = list(body_pipe._modules.keys())
        print(f"  Loaded modules: {len(loaded_mods)}/10 — {loaded_mods}")
        body_output = body_pipe.process(video_path)
        body_time = time.time() - t2
        print(f"  ✓ Body done in {body_time:.0f}s")
        print(f"  Segments: {len(body_output.get('segments', []))}")
        results["body"] = {
            "status": "OK",
            "time_sec": body_time,
            "segment_count": len(body_output.get("segments", [])),
            "modules_loaded": len(loaded_mods),
            "posture_score": body_output.get("summary", {}).get("posture_score", 0),
        }
    except Exception as e:
        print(f"  ✗ Body FAILED: {e}")
        traceback.print_exc()
        body_output = {"segments": [], "summary": {}}
        results["body"] = {"status": "FAILED", "error": str(e)}
        results["errors"].append(f"Body: {e}")

    torch.cuda.empty_cache()

    # ── Modality 3: Content ────────────────────────────────────
    try:
        print("\n[3/8] Content analysis...")
        t3 = time.time()
        from src.content.pipeline import ContentAnalysisPipeline
        content_pipe = ContentAnalysisPipeline()
        # Pass the transcript dict directly (not the full voice_output)
        transcript = voice_output.get("transcript", {})
        if not transcript or not isinstance(transcript, dict):
            print("  WARNING: No transcript from voice pipeline, using empty")
            transcript = {"text": "", "words": []}
        else:
            print(f"  Transcript: {len(transcript.get('words', []))} words")
        content_output = content_pipe.process(transcript)
        content_time = time.time() - t3
        print(f"  ✓ Content done in {content_time:.0f}s")
        print(f"  Segments: {len(content_output.get('segments', []))}")
        results["content"] = {
            "status": "OK",
            "time_sec": content_time,
            "segment_count": len(content_output.get("segments", [])),
        }
    except Exception as e:
        print(f"  ✗ Content FAILED: {e}")
        traceback.print_exc()
        content_output = {"segments": [], "summary": {}}
        results["content"] = {"status": "FAILED", "error": str(e)}
        results["errors"].append(f"Content: {e}")

    # ── Fusion + Scoring + Report (CPU, no GPU) ────────────────
    try:
        print("\n[4/8] Multimodal fusion...")
        duration = (
            voice_output.get("summary", {}).get("duration_sec")
            or body_output.get("duration_sec")
            or 300
        )

        voice_windows = voice_output.get("windows", [])
        if isinstance(voice_output, list):
            voice_windows = voice_output

        from src.fusion.fusion_engine import FusionEngine
        engine = FusionEngine()
        fusion_output = engine.fuse(voice_windows, body_output, content_output, duration)
        print(f"  ✓ Timeline: {len(fusion_output['timeline'])} seconds")
        print(f"  ✓ Boundaries: {len(fusion_output['regime_boundaries'])}")
        print(f"  ✓ Disruptions: {fusion_output['recovery']['total_disruptions']}")
        print(f"  ✓ Coherence: {fusion_output['emotion_coherence']['coherence_score_0_100']}/100")

        print("\n[5/8] Computing scores...")
        from src.scoring.dimension_scorer import DimensionScorer
        scorer = DimensionScorer()
        scores = scorer.score_all(fusion_output)
        print(f"  Overall: {scores['overall']:.0f}/100")
        for dim, val in scores.items():
            if dim != "overall":
                print(f"  {dim}: {val:.0f}/100")

        print("\n[6/8] Generating coaching feedback...")
        from src.report.coaching_writer import CoachingWriter
        writer = CoachingWriter(use_api=False)  # Template mode on Colab
        coaching = writer.generate_full_report(
            all_segments=[],
            overall_scores=scores,
            fusion_output=fusion_output,
            speech_metadata={"duration_sec": duration, "video_path": video_path},
        )

        print("\n[7/8] Generating charts...")
        from src.report.chart_generator import ChartGenerator
        chart_dir = os.path.join(output_dir, "charts")
        gen = ChartGenerator(output_dir=chart_dir, dpi=150)
        charts = gen.generate_all(scores, fusion_output["timeline"], fusion_output)
        print(f"  ✓ {len(charts)} charts generated")

        print("\n[8/8] Building report...")
        from src.report.report_builder import ReportBuilder
        builder = ReportBuilder()
        report = builder.build_and_save(
            coaching_data=coaching,
            scores=scores,
            charts=charts,
            metadata={
                "duration_sec": duration,
                "video_path": video_path,
                "processing_time_sec": time.time() - t0,
            },
            output_dir=output_dir,
        )

        results["fusion"] = {
            "status": "OK",
            "timeline_length": len(fusion_output["timeline"]),
            "boundaries": len(fusion_output["regime_boundaries"]),
            "disruptions": fusion_output["recovery"]["total_disruptions"],
            "coherence": fusion_output["emotion_coherence"]["coherence_score_0_100"],
        }
        results["scores"] = scores
        results["report"] = report

    except Exception as e:
        print(f"  ✗ Fusion/Report FAILED: {e}")
        traceback.print_exc()
        results["errors"].append(f"Fusion/Report: {e}")

    total_time = time.time() - t0
    results["total_time_sec"] = total_time
    print(f"\n{'='*60}")
    print(f"[{label.upper()}] COMPLETE in {total_time:.0f}s")
    if results.get("report"):
        print(f"HTML: {results['report']['html']}")
    if results["errors"]:
        print(f"ERRORS: {results['errors']}")
    print(f"{'='*60}")

    # Save results JSON
    results_path = os.path.join(output_dir, "validation_results.json")
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2, default=str)

    return results

print("✓ Pipeline runner defined")

## Test A: Good Speaker (TED Talk, ~5 min)
Expected: Most dimension scores 70-90/100

In [ ]:
# Cell 7: Test A — Good Speaker
results_good = run_full_pipeline(
    "data/raw/test_good_speaker_5min.mp4",
    label="test_good",
    output_base="reports"
)

## Test B: Nervous Speaker (Student Presentation, ~5 min)
Expected: Some dimension scores 40-60/100, disruptions detected

In [ ]:
# Cell 8: Test B — Nervous Speaker
results_nervous = run_full_pipeline(
    "data/raw/test_nervous_speaker_5min.mp4",
    label="test_nervous",
    output_base="reports"
)

## Test C: Monotone Speaker (Dry Lecture, ~5 min)
Expected: Emotional Expressiveness low (<50), Content Structure potentially high

In [ ]:
# Cell 9: Test C — Monotone Speaker
results_monotone = run_full_pipeline(
    "data/raw/test_monotone_speaker_5min.mp4",
    label="test_monotone",
    output_base="reports"
)

## Validation Summary

In [ ]:
# Cell 10: Validation Summary Table
import json

all_results = [results_good, results_nervous, results_monotone]

print("=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)

# Modality status
print("\n── Modality Pipeline Status ──")
print(f"{'Test':<12} {'Voice':<10} {'Body':<10} {'Content':<10} {'Fusion':<10} {'Time':<8}")
print("-" * 60)
for r in all_results:
    v = r.get("voice", {}).get("status", "N/A")
    b = r.get("body", {}).get("status", "N/A")
    c = r.get("content", {}).get("status", "N/A")
    f = r.get("fusion", {}).get("status", "N/A")
    t = r.get("total_time_sec", 0)
    print(f"{r['label']:<12} {v:<10} {b:<10} {c:<10} {f:<10} {t:.0f}s")

# Scores comparison
print("\n── Dimension Scores ──")
dims = ["vocal_clarity", "body_language", "content_structure",
        "audience_engagement", "emotional_expressiveness",
        "regime_adaptability", "overall"]

header = f"{'Dimension':<28}"
for r in all_results:
    header += f" {r['label']:<14}"
print(header)
print("-" * 70)

for dim in dims:
    row = f"{dim.replace('_', ' ').title():<28}"
    for r in all_results:
        s = r.get("scores", {}).get(dim, "-")
        if isinstance(s, (int, float)):
            row += f" {s:>6.0f}/100    "
        else:
            row += f" {'N/A':>10}    "
    print(row)

# Fusion details
print("\n── Fusion Details ──")
print(f"{'Metric':<28}", end="")
for r in all_results:
    print(f" {r['label']:<14}", end="")
print()
print("-" * 70)

for metric in ["timeline_length", "boundaries", "disruptions", "coherence"]:
    row = f"{metric:<28}"
    for r in all_results:
        val = r.get("fusion", {}).get(metric, "-")
        row += f" {str(val):>10}    "
    print(row)

# Errors
print("\n── Errors ──")
total_errors = sum(len(r.get("errors", [])) for r in all_results)
if total_errors == 0:
    print("✓ No errors across all tests!")
else:
    for r in all_results:
        if r.get("errors"):
            for e in r["errors"]:
                print(f"  ✗ [{r['label']}] {e}")

# Reports
print("\n── Generated Reports ──")
for r in all_results:
    report = r.get("report", {})
    html = report.get("html", "N/A")
    pdf = report.get("pdf", "N/A")
    print(f"  {r['label']}: HTML={html}")
    if pdf and pdf != "N/A":
        print(f"  {' '*len(r['label'])}  PDF={pdf}")

print(f"\n{'='*70}")
if total_errors == 0:
    print("✓ ALL TESTS PASSED — Pipeline validated end-to-end on T4 GPU")
else:
    print(f"⚠ {total_errors} error(s) detected — see details above")
print(f"{'='*70}")

In [ ]:
# Cell 11: View radar charts side-by-side
from IPython.display import display, Image, HTML
import os

print("Radar Charts — Score Comparison\n")
for r in all_results:
    label = r["label"]
    chart_path = os.path.join("reports", label, "charts", "radar_chart.png")
    if os.path.exists(chart_path):
        print(f"── {label} ──")
        display(Image(filename=chart_path, width=400))
    else:
        print(f"  {label}: chart not found")

In [ ]:
# Cell 12: View HTML report in notebook (good speaker)
from IPython.display import IFrame
import os

html_path = "reports/test_good/speech_report.html"
if os.path.exists(html_path):
    display(IFrame(src=html_path, width="100%", height=800))
else:
    print(f"Report not found at {html_path}")

In [ ]:
# Cell 13: Copy reports back to Drive for download
import shutil, os

drive_reports = "/content/drive/MyDrive/Claude-assistant/reports"
local_reports = "/content/Claude-assistant/reports"

if os.path.exists(local_reports):
    shutil.copytree(local_reports, drive_reports, dirs_exist_ok=True)
    print(f"✓ Reports copied to Drive: {drive_reports}")
    for root, dirs, files in os.walk(drive_reports):
        for f in files:
            path = os.path.join(root, f)
            size = os.path.getsize(path) / 1024
            rel = os.path.relpath(path, drive_reports)
            print(f"  {rel} ({size:.0f} KB)")
else:
    print("No reports directory found")

In [ ]:
# Cell 14: Final validation checklist
print("""
╔══════════════════════════════════════════════════════════════╗
║                                                              ║
║   AI PUBLIC SPEAKING ASSISTANT — VALIDATION CHECKLIST        ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║   MODALITY 1 (Voice):                                       ║
║   [ ] Whisper produced word-level timestamps                ║
║   [ ] Filler words detected with timestamps                 ║
║   [ ] Prosody metrics in reasonable ranges                  ║
║   [ ] Vocal emotion varies across speech                    ║
║                                                              ║
║   MODALITY 2 (Body):                                        ║
║   [ ] Person detected in all frames                         ║
║   [ ] Pose keypoints extracted                              ║
║   [ ] Gestures classified                                   ║
║   [ ] Gaze zones computed                                   ║
║                                                              ║
║   MODALITY 3 (Content):                                     ║
║   [ ] Grammar errors detected (not false positives)         ║
║   [ ] Readability in expected range (FKGL 8-14)            ║
║   [ ] Regime boundaries at logical transition points        ║
║   [ ] Sentiment matches content                             ║
║                                                              ║
║   FUSION:                                                    ║
║   [ ] Timeline aligned at 1-second resolution              ║
║   [ ] Regime transitions scored                             ║
║   [ ] Recovery events detected (if disruptions exist)       ║
║   [ ] Emotion coherence computed                            ║
║                                                              ║
║   REPORT:                                                    ║
║   [ ] Executive summary reads like a human coach            ║
║   [ ] Radar chart shows 6 scores                            ║
║   [ ] Timeline heatmap renders correctly                    ║
║   [ ] Per-segment coaching is specific with timestamps      ║
║   [ ] Practice plan has 3 actionable items                  ║
║   [ ] HTML opens correctly in browser                       ║
║                                                              ║
║   PERFORMANCE:                                               ║
║   [ ] Full pipeline completes in < 10 minutes on T4         ║
║   [ ] No OOM errors                                         ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

# Auto-check what we can
total_errors = sum(len(r.get("errors", [])) for r in all_results)
all_ok = all(
    r.get("voice", {}).get("status") == "OK" and
    r.get("body", {}).get("status") == "OK" and
    r.get("content", {}).get("status") == "OK" and
    r.get("fusion", {}).get("status") == "OK"
    for r in all_results
)

if all_ok and total_errors == 0:
    print("""
╔══════════════════════════════════════════════════════════════╗
║                                                              ║
║   AI PUBLIC SPEAKING ASSISTANT — BUILD COMPLETE              ║
║                                                              ║
║   All 3 modalities ✓                                         ║
║   Multimodal fusion ✓                                        ║
║   6-dimension scoring ✓                                      ║
║   Coaching report generation ✓                               ║
║   HTML report output ✓                                       ║
║                                                              ║
║   Usage:                                                     ║
║   python -m src.master_pipeline --video YOUR_SPEECH.mp4      ║
║                                                              ║
║   Report: reports/speech_report.html                         ║
║   Cost per speech: $0.00 (via Max subscription proxy)        ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")
else:
    print(f"\n⚠ {total_errors} error(s) — review output above before marking complete.")